# Representasi Permutasi Latin Square

Program ini:
1. Generate semua Latin square order $n$
2. Ekstrak representasi permutasi untuk setiap elemen (sesuai Proposisi 2.6.1)
3. Export hasil ke file LaTeX

**Proposisi 2.6.1**: Setiap Latin square $A \in L_S(n)$ dengan simbol $\{a_1, a_2, \ldots, a_n\}$ dapat dituliskan sebagai:
$$A = \bigoplus_{i=1}^{n} a_i P_{\tau_i}$$
dimana $P_{\tau_i}$ adalah matriks permutasi max-plus yang bersesuaian dengan permutasi $\tau_i$.

In [43]:
"""
Latin Square Library - Reusable untuk berbagai order
"""
import numpy as np
from typing import List, Dict, Tuple
import os

class LatinSquare:
    """Class untuk merepresentasikan Latin Square dan operasinya"""
    
    def __init__(self, matrix: np.ndarray, symbols: List[int] = None):
        self.matrix = np.array(matrix, dtype=int)
        self.n = len(matrix)
        self.symbols = symbols if symbols else sorted(np.unique(matrix).tolist())
    
    def __repr__(self):
        return f"LatinSquare(n={self.n})\n{self.matrix}"
    
    def __eq__(self, other):
        return np.array_equal(self.matrix, other.matrix)
    
    def __hash__(self):
        return hash(self.matrix.tobytes())

class LatinSquareGenerator:
    """Generator Latin Square menggunakan backtracking dengan constraint propagation"""
    
    def __init__(self, n: int, symbols: List[int] = None):
        self.n = n
        self.symbols = symbols if symbols else list(range(1, n + 1))
        self.all_squares = []
    
    def generate_all(self) -> List[LatinSquare]:
        """Generate semua Latin square order n dengan constraint propagation"""
        self.all_squares = []
        
        # Initialize empty grid dengan possible values
        grid = [[0] * self.n for _ in range(self.n)]
        possible = [[set(self.symbols) for _ in range(self.n)] for _ in range(self.n)]
        
        self._backtrack(grid, possible, 0, 0)
        
        return self.all_squares
    
    def _backtrack(self, grid, possible, row, col):
        """Backtracking dengan constraint propagation"""
        if row == self.n:
            # Found valid Latin square
            self.all_squares.append(
                LatinSquare(np.array(grid), self.symbols.copy())
            )
            return
        
        next_row, next_col = (row, col + 1) if col + 1 < self.n else (row + 1, 0)
        
        for val in list(possible[row][col]):
            # Place value
            grid[row][col] = val
            
            # Save state for backtracking
            removed = []
            
            # Propagate constraint: remove val from same row and column
            valid = True
            for k in range(self.n):
                if k != col and val in possible[row][k]:
                    possible[row][k].remove(val)
                    removed.append((row, k, val))
                    if not possible[row][k] and grid[row][k] == 0:
                        valid = False
                if k != row and val in possible[k][col]:
                    possible[k][col].remove(val)
                    removed.append((k, col, val))
                    if not possible[k][col] and grid[k][col] == 0:
                        valid = False
            
            if valid:
                self._backtrack(grid, possible, next_row, next_col)
            
            # Restore state
            grid[row][col] = 0
            for r, c, v in removed:
                possible[r][c].add(v)
    
    def count(self) -> int:
        """Return jumlah Latin square yang sudah di-generate"""
        return len(self.all_squares)

print("✓ Library LatinSquare berhasil dimuat!")
print("  - LatinSquare: Class untuk representasi Latin square")
print("  - LatinSquareGenerator: Class untuk generate semua Latin square")

✓ Library LatinSquare berhasil dimuat!
  - LatinSquare: Class untuk representasi Latin square
  - LatinSquareGenerator: Class untuk generate semua Latin square


In [44]:
class PermutationExtractor:
    """
    Ekstrak representasi permutasi dari Latin square (Proposisi 2.6.1)
    
    Untuk setiap simbol a_i, definisikan permutasi τ_i dimana:
        τ_i(r) = c  ⟺  [A]_rc = a_i
    
    Artinya: τ_i memetakan baris r ke kolom c dimana elemen a_i berada
    """
    
    @staticmethod
    def extract_permutations(ls: LatinSquare) -> Dict[int, Tuple[int, ...]]:
        """
        Ekstrak permutasi untuk setiap simbol dalam Latin square
        
        Returns:
            Dict mapping simbol -> permutasi (dalam bentuk tuple)
            Permutasi τ[r] = c berarti simbol berada di kolom c untuk baris r
        """
        n = ls.n
        permutations_dict = {}
        
        for symbol in ls.symbols:
            perm = [0] * n  # τ(0), τ(1), ..., τ(n-1)
            for r in range(n):
                for c in range(n):
                    if ls.matrix[r, c] == symbol:
                        perm[r] = c
                        break
            permutations_dict[symbol] = tuple(perm)
        
        return permutations_dict
    
    @staticmethod
    def extract_permutation_matrices(ls: LatinSquare) -> Dict[int, np.ndarray]:
        """
        Ekstrak matriks permutasi untuk setiap simbol
        
        Returns:
            Dict mapping simbol -> matriks permutasi P_τ
            P_τ[i,j] = 1 jika τ(i) = j, 0 otherwise
        """
        n = ls.n
        perm_matrices = {}
        
        for symbol in ls.symbols:
            P = np.zeros((n, n), dtype=int)
            for r in range(n):
                for c in range(n):
                    if ls.matrix[r, c] == symbol:
                        P[r, c] = 1
            perm_matrices[symbol] = P
        
        return perm_matrices
    
    @staticmethod
    def permutation_to_cycle_notation(perm: Tuple[int, ...], one_indexed: bool = True) -> str:
        """
        Convert permutasi ke notasi siklus
        
        Args:
            perm: Tuple permutasi dimana perm[i] = j berarti i -> j
            one_indexed: Jika True, gunakan 1,2,3,... (untuk LaTeX)
        
        Returns:
            String notasi siklus, misal "(1 2 3)" atau "(1 3)(2)"
        """
        n = len(perm)
        offset = 1 if one_indexed else 0
        visited = [False] * n
        cycles = []
        
        for start in range(n):
            if visited[start]:
                continue
            
            cycle = []
            current = start
            while not visited[current]:
                visited[current] = True
                cycle.append(current + offset)
                current = perm[current]
            
            if len(cycle) > 1:
                cycles.append(cycle)
            elif len(cycle) == 1:
                # Fixed point - bisa ditampilkan atau tidak
                cycles.append(cycle)
        
        if not cycles:
            return "()"
        
        return "".join(f"({' '.join(map(str, c))})" for c in cycles)

print("✓ PermutationExtractor berhasil dimuat!")
print("  - extract_permutations: Ekstrak permutasi dari Latin square")
print("  - extract_permutation_matrices: Ekstrak matriks permutasi")
print("  - permutation_to_cycle_notation: Konversi ke notasi siklus")

✓ PermutationExtractor berhasil dimuat!
  - extract_permutations: Ekstrak permutasi dari Latin square
  - extract_permutation_matrices: Ekstrak matriks permutasi
  - permutation_to_cycle_notation: Konversi ke notasi siklus


In [45]:
class LaTeXExporter:
    """Export Latin square dan representasi permutasinya ke LaTeX (format ringkas untuk twocolumn)"""
    
    def __init__(self, output_dir: str = "."):
        self.output_dir = output_dir
    
    def matrix_to_latex(self, matrix: np.ndarray, env: str = "pmatrix") -> str:
        """Convert numpy matrix ke LaTeX matrix environment"""
        n = matrix.shape[0]
        rows = []
        for i in range(n):
            row_str = " & ".join(str(int(matrix[i, j])) for j in range(n))
            rows.append(row_str)
        
        content = " \\\\ ".join(rows)
        return f"\\begin{{{env}}} {content} \\end{{{env}}}"
    
    def cycle_notation_to_latex(self, perm: Tuple[int, ...], one_indexed: bool = True) -> str:
        """Convert permutasi ke notasi siklus LaTeX"""
        return PermutationExtractor.permutation_to_cycle_notation(perm, one_indexed)
    
    def export_latin_square(self, ls: LatinSquare, index: int) -> str:
        """Export satu Latin square ke LaTeX (format kompak dengan matrix untuk permutasi)"""
        perms = PermutationExtractor.extract_permutations(ls)
        
        # Buat matrix untuk permutasi: tau_1 \\ tau_2 \\ tau_3
        perm_rows = " \\\\ ".join(
            f"\\tau_{{{s}}} = {self.cycle_notation_to_latex(perms[s])}" 
            for s in ls.symbols
        )
        
        # Format: L_i = matrix, begin{matrix} permutasi \end{matrix}
        return f"$L_{{{index}}} = {self.matrix_to_latex(ls.matrix)}, \\quad \\begin{{matrix}} {perm_rows} \\end{{matrix}}$\n"
    
    def export_all(self, squares: List[LatinSquare], 
                   filename: str,
                   title: str = None) -> str:
        """Export semua Latin square ke file .tex (format ringkas)"""
        lines = []
        
        n = squares[0].n if squares else 0
        symbols_str = ", ".join(map(str, squares[0].symbols)) if squares else ""
        
        # Header comment
        lines.append(f"% File: {filename}.tex")
        lines.append(f"% Latin Squares order {n}, symbols {{{symbols_str}}}, total: {len(squares)}")
        lines.append("")
        
        if title:
            lines.append(f"\\subsection*{{{title}}}")
        else:
            lines.append(f"\\subsection*{{Latin Square Order {n}}}")
        lines.append("")
        
        lines.append(f"Terdapat {len(squares)} Latin square berbeda dengan order {n} dan simbol $\\{{{symbols_str}\\}}$:")
        lines.append("")
        
        # Export each Latin square
        for i, ls in enumerate(squares, 1):
            lines.append(self.export_latin_square(ls, i))
        
        # Write to file
        filepath = os.path.join(self.output_dir, f"{filename}.tex")
        with open(filepath, 'w', encoding='utf-8') as f:
            f.write("\n".join(lines))
        
        return filepath

print("✓ LaTeXExporter berhasil dimuat (format kompak untuk twocolumn)!")

✓ LaTeXExporter berhasil dimuat (format kompak untuk twocolumn)!


## Contoh: Generate Latin Square Order 3

Menggunakan simbol $\{1, 2, 3\}$

In [46]:
# ============================================================
# KONFIGURASI - Ubah ini untuk order yang berbeda
# ============================================================
ORDER = 5
SYMBOLS = list(range(1, ORDER + 1)) 

# ============================================================
# Generate semua Latin square
# ============================================================
print(f"{'='*60}")
print(f"GENERATING LATIN SQUARES ORDER {ORDER}")
print(f"Simbol: {{{', '.join(map(str, SYMBOLS))}}}")
print(f"{'='*60}")

generator = LatinSquareGenerator(ORDER, SYMBOLS)
all_squares = generator.generate_all()

print(f"\n✓ Ditemukan {len(all_squares)} Latin square")
print(f"\nContoh Latin square pertama:")
print(all_squares[0].matrix)

GENERATING LATIN SQUARES ORDER 5
Simbol: {1, 2, 3, 4, 5}

✓ Ditemukan 161280 Latin square

Contoh Latin square pertama:
[[1 2 3 4 5]
 [2 1 4 5 3]
 [3 4 5 2 1]
 [4 5 1 3 2]
 [5 3 2 1 4]]


## Export ke LaTeX

Export hasil ke file `.tex` yang bisa di-import dengan `\input{}`

In [47]:
# ============================================================
# Export ke LaTeX
# ============================================================
exporter = LaTeXExporter(output_dir=".")

output_file = f"latin_square_order_{ORDER}"
filepath = exporter.export_all(
    all_squares,
    filename=output_file,
    title=f"Latin Square Order {ORDER}"
)

print(f"✓ File LaTeX berhasil dibuat: {filepath}")
print(f"  Total: {len(all_squares)} Latin square")
print(f"\nUntuk import di main.tex, gunakan:")
print(f"  \\input{{{output_file}}}")

✓ File LaTeX berhasil dibuat: .\latin_square_order_5.tex
  Total: 161280 Latin square

Untuk import di main.tex, gunakan:
  \input{latin_square_order_5}


In [48]:
# ============================================================
# Preview isi file LaTeX yang dihasilkan
# ============================================================
print("="*60)
print("PREVIEW FILE LATEX")
print("="*60)

with open(filepath, 'r', encoding='utf-8') as f:
    content = f.read()
    # Tampilkan 80 baris pertama
    lines = content.split('\n')
    preview_lines = lines[:80]
    print('\n'.join(preview_lines))
    if len(lines) > 80:
        print(f"\n... ({len(lines) - 80} baris lagi)")

PREVIEW FILE LATEX
% File: latin_square_order_5.tex
% Latin Squares order 5, symbols {1, 2, 3, 4, 5}, total: 161280

\subsection*{Latin Square Order 5}

Terdapat 161280 Latin square berbeda dengan order 5 dan simbol $\{1, 2, 3, 4, 5\}$:

$L_{1} = \begin{pmatrix} 1 & 2 & 3 & 4 & 5 \\ 2 & 1 & 4 & 5 & 3 \\ 3 & 4 & 5 & 2 & 1 \\ 4 & 5 & 1 & 3 & 2 \\ 5 & 3 & 2 & 1 & 4 \end{pmatrix}, \quad \begin{matrix} \tau_{1} = (1)(2)(3 5 4) \\ \tau_{2} = (1 2)(3 4 5) \\ \tau_{3} = (1 3)(2 5)(4) \\ \tau_{4} = (1 4)(2 3)(5) \\ \tau_{5} = (1 5)(2 4)(3) \end{matrix}$

$L_{2} = \begin{pmatrix} 1 & 2 & 3 & 4 & 5 \\ 2 & 1 & 4 & 5 & 3 \\ 3 & 4 & 5 & 2 & 1 \\ 5 & 3 & 2 & 1 & 4 \\ 4 & 5 & 1 & 3 & 2 \end{pmatrix}, \quad \begin{matrix} \tau_{1} = (1)(2)(3 5)(4) \\ \tau_{2} = (1 2)(3 4)(5) \\ \tau_{3} = (1 3)(2 5 4) \\ \tau_{4} = (1 4 5)(2 3) \\ \tau_{5} = (1 5 2 4)(3) \end{matrix}$

$L_{3} = \begin{pmatrix} 1 & 2 & 3 & 4 & 5 \\ 2 & 1 & 4 & 5 & 3 \\ 3 & 4 & 5 & 1 & 2 \\ 4 & 5 & 2 & 3 & 1 \\ 5 & 3 & 1 & 2 & 4 \end{pma

## Cara Menggunakan untuk Order Lain

Untuk generate Latin square dengan order berbeda, cukup ubah nilai `ORDER` di cell konfigurasi. Contoh:

```python
# Untuk order 4
ORDER = 4
SYMBOLS = list(range(1, ORDER + 1))  # [1, 2, 3, 4]

# Untuk order 5 (membutuhkan waktu lebih lama)
ORDER = 5
SYMBOLS = list(range(1, ORDER + 1))  # [1, 2, 3, 4, 5]
```

**Jumlah Latin Square per Order:**

| Order | Jumlah |
|-------|--------|
| 1 | 1 |
| 2 | 2 |
| 3 | 12 |
| 4 | 576 |
| 5 | 161,280 |

## Pasangan Latin Square Komutatif (Max-Plus) — Optimized

Mencari semua pasangan $(A, B)$ dengan $A \neq B$ yang memenuhi: $A \otimes B = B \otimes A$

### Optimasi: Filter Permutasi Elemen Maksimum

**Syarat Perlu (Necessary Condition):** Jika $A \otimes B = B \otimes A$, maka $\sigma_n^A \circ \sigma_n^B = \sigma_n^B \circ \sigma_n^A$

dimana $\sigma_n^X$ adalah permutasi simbol maksimum $n$ di Latin square $X$.

**Bukti singkat:** $(A \otimes B)_{ij} = \max_k(A_{ik} + B_{kj})$. Entri dengan $A_{ik} = n$ dan $B_{kj} = n$ memberikan nilai $2n$ (nilai maks global). Posisi entri $2n$ di $A \otimes B$ ditentukan oleh $\sigma_n^B \circ \sigma_n^A$, sedangkan di $B \otimes A$ oleh $\sigma_n^A \circ \sigma_n^B$. Kesamaan matriks mengharuskan kedua komposisi ini sama. $\square$

**Strategi:**
1. Kelompokkan Latin square berdasarkan permutasi simbol maksimumnya
2. Hanya cek pasangan dari grup yang permutasinya komutatif
3. Gunakan batch numpy vectorized max-plus multiplication

Untuk $S_5$, hanya ~5% pasangan permutasi yang komutatif → **reduksi ~95%** dari total pasangan.

In [49]:
# ============================================================
# OPTIMIZED: Max-Plus Commutativity Check
# Filter: Permutasi elemen maksimum harus komutatif
# + Streaming batch (no meshgrid, no large index arrays)
# ============================================================
import time
import gc
from collections import defaultdict

# ── Fungsi Max-Plus (Memory-Efficient) ───────────────────────

def maxplus_mult(A: np.ndarray, B: np.ndarray) -> np.ndarray:
    """Perkalian max-plus single pair — vectorized numpy"""
    n = A.shape[0]
    result = A[:, 0:1] + B[0:1, :]
    for k in range(1, n):
        np.maximum(result, A[:, k:k+1] + B[k:k+1, :], out=result)
    return result

def maxplus_mult_batch(A_batch: np.ndarray, B_batch: np.ndarray) -> np.ndarray:
    """
    Batch max-plus — memory-efficient (loop over k, no 4D tensor).
    Memory: O(batch × n²) instead of O(batch × n³).
    """
    n = A_batch.shape[1]
    result = A_batch[:, :, 0:1] + B_batch[:, 0:1, :]
    for k in range(1, n):
        np.maximum(result, A_batch[:, :, k:k+1] + B_batch[:, k:k+1, :], out=result)
    return result

# ── Fungsi Permutasi ────────────────────────────────────────

def compose_perm(p1: tuple, p2: tuple) -> tuple:
    return tuple(p1[p2[i]] for i in range(len(p1)))

def extract_max_perm(ls: LatinSquare) -> tuple:
    max_sym = max(ls.symbols)
    n = ls.n
    return tuple(
        next(c for c in range(n) if ls.matrix[r, c] == max_sym)
        for r in range(n)
    )

# ── Streaming pair generators (no large arrays) ─────────────

def _pairs_same_group(idx_list, batch_size):
    """Yield chunks of (a_indices, b_indices) for pairs within one group."""
    m = len(idx_list)
    buf_a, buf_b = [], []
    for i in range(m):
        for j in range(i + 1, m):
            buf_a.append(idx_list[i])
            buf_b.append(idx_list[j])
            if len(buf_a) >= batch_size:
                yield np.array(buf_a, dtype=np.int32), np.array(buf_b, dtype=np.int32)
                buf_a, buf_b = [], []
    if buf_a:
        yield np.array(buf_a, dtype=np.int32), np.array(buf_b, dtype=np.int32)

def _pairs_cross_group(idx_i, idx_j, batch_size):
    """Yield chunks of (a_indices, b_indices) for pairs across two groups."""
    buf_a, buf_b = [], []
    for a in idx_i:
        for b in idx_j:
            buf_a.append(a)
            buf_b.append(b)
            if len(buf_a) >= batch_size:
                yield np.array(buf_a, dtype=np.int32), np.array(buf_b, dtype=np.int32)
                buf_a, buf_b = [], []
    if buf_a:
        yield np.array(buf_a, dtype=np.int32), np.array(buf_b, dtype=np.int32)

# ── Pencarian Pasangan Komutatif (Optimized) ────────────────

def find_commutative_pairs(squares: List[LatinSquare],
                           batch_size: int = 2000) -> List[dict]:
    """
    Cari pasangan komutatif A ⊗ B = B ⊗ A dengan optimasi:
    
    1. FILTER: σ_n^A ∘ σ_n^B = σ_n^B ∘ σ_n^A (syarat perlu)
    2. STREAMING: pair indices generated on-the-fly (no meshgrid)
    3. BATCH numpy max-plus (memory-efficient, no 4D tensor)
    4. int8 dtype
    """
    n_sq = len(squares)
    if n_sq == 0:
        return []
    
    start_time = time.time()
    total_all = n_sq * (n_sq - 1) // 2
    
    # ═══════════════════════════════════════════════════════════
    # LANGKAH 1: Ekstrak permutasi simbol maksimum
    # ═══════════════════════════════════════════════════════════
    print(f"\n  [1/4] Mengekstrak permutasi simbol maksimum...")
    max_perms = [extract_max_perm(ls) for ls in squares]
    
    # ═══════════════════════════════════════════════════════════
    # LANGKAH 2: Kelompokkan berdasarkan permutasi
    # ═══════════════════════════════════════════════════════════
    groups = defaultdict(list)
    for idx, perm in enumerate(max_perms):
        groups[perm].append(idx)
    del max_perms
    gc.collect()
    
    perm_keys = list(groups.keys())
    n_groups = len(perm_keys)
    sizes = [len(groups[p]) for p in perm_keys]
    
    print(f"  [2/4] {n_groups} grup permutasi ditemukan")
    print(f"         Ukuran: min={min(sizes)}, max={max(sizes)}, "
          f"mean={np.mean(sizes):.0f}")
    
    # ═══════════════════════════════════════════════════════════
    # LANGKAH 3: Filter — cari pasangan grup yang komutatif
    # ═══════════════════════════════════════════════════════════
    comm_gpairs = []
    for i in range(n_groups):
        for j in range(i, n_groups):
            if compose_perm(perm_keys[i], perm_keys[j]) == \
               compose_perm(perm_keys[j], perm_keys[i]):
                comm_gpairs.append((i, j))
    
    total_filtered = sum(
        len(groups[perm_keys[gi]]) * (len(groups[perm_keys[gi]]) - 1) // 2 
        if gi == gj else
        len(groups[perm_keys[gi]]) * len(groups[perm_keys[gj]])
        for gi, gj in comm_gpairs
    )
    
    red_pct = (1 - total_filtered / total_all) * 100 if total_all > 0 else 0
    print(f"  [3/4] {len(comm_gpairs)} pasangan grup komutatif")
    print(f"         Pasangan lolos filter: {total_filtered:,} / {total_all:,}")
    print(f"         Reduksi: {red_pct:.1f}% tereliminasi")
    
    # ═══════════════════════════════════════════════════════════
    # LANGKAH 4: Streaming batch max-plus check
    # ═══════════════════════════════════════════════════════════
    print(f"  [4/4] Streaming batch max-plus check (batch_size={batch_size:,})...")
    
    # Pre-konversi ke int8 (values 1..n, max sum 2n ≤ 127)
    mats = np.array([ls.matrix for ls in squares], dtype=np.int8)
    gc.collect()
    
    commutative_pairs = []
    total_checked = 0
    last_report = time.time()
    
    for gp_idx, (gi, gj) in enumerate(comm_gpairs):
        idx_i = groups[perm_keys[gi]]
        idx_j = groups[perm_keys[gj]]
        
        # Streaming pair generator — no large index arrays
        if gi == gj:
            pair_gen = _pairs_same_group(idx_i, batch_size)
        else:
            pair_gen = _pairs_cross_group(idx_i, idx_j, batch_size)
        
        for chunk_a, chunk_b in pair_gen:
            A_batch = mats[chunk_a]
            B_batch = mats[chunk_b]
            
            AB = maxplus_mult_batch(A_batch, B_batch)
            BA = maxplus_mult_batch(B_batch, A_batch)
            
            # Cek kesamaan (vectorized)
            equal_mask = np.all(AB == BA, axis=(1, 2))
            
            # Simpan pasangan komutatif
            for k in np.where(equal_mask)[0]:
                ia, ib = int(chunk_a[k]), int(chunk_b[k])
                if ia < ib:
                    commutative_pairs.append({
                        'A': squares[ia], 'B': squares[ib],
                        'A_index': ia + 1, 'B_index': ib + 1,
                        'product': AB[k].astype(int)
                    })
                else:
                    commutative_pairs.append({
                        'A': squares[ib], 'B': squares[ia],
                        'A_index': ib + 1, 'B_index': ia + 1,
                        'product': BA[k].astype(int)
                    })
            
            total_checked += len(chunk_a)
            del A_batch, B_batch, AB, BA, equal_mask
        
        # Progress report setiap 10 detik
        now = time.time()
        if now - last_report > 10:
            elapsed = now - start_time
            pct = 100 * total_checked / total_filtered if total_filtered > 0 else 100
            rate = total_checked / elapsed if elapsed > 0 else 1
            eta = (total_filtered - total_checked) / rate if rate > 0 else 0
            print(f"         {total_checked:,}/{total_filtered:,} ({pct:.1f}%) "
                  f"| {elapsed:.0f}s elapsed | ETA ~{eta:.0f}s "
                  f"| {len(commutative_pairs)} found")
            last_report = now
    
    elapsed = time.time() - start_time
    print(f"\n  ✓ Selesai dalam {elapsed:.1f} detik")
    
    del mats
    gc.collect()
    
    return commutative_pairs

# ════════════════════════════════════════════════════════════════
# Jalankan pencarian
# ════════════════════════════════════════════════════════════════
print(f"{'='*60}")
print(f"MENCARI PASANGAN KOMUTATIF (A ⊗ B = B ⊗ A)")
print(f"{'='*60}")
print(f"Total Latin square: {len(all_squares)}")
print(f"Total pasangan brute-force: {len(all_squares) * (len(all_squares) - 1) // 2:,}")

commutative_pairs = find_commutative_pairs(all_squares)

print(f"\n{'='*60}")
print(f"✓ Ditemukan {len(commutative_pairs)} pasangan komutatif")
print(f"  dari total {len(all_squares) * (len(all_squares) - 1) // 2:,} pasangan")

MENCARI PASANGAN KOMUTATIF (A ⊗ B = B ⊗ A)
Total Latin square: 161280
Total pasangan brute-force: 13,005,538,560

  [1/4] Mengekstrak permutasi simbol maksimum...
  [2/4] 120 grup permutasi ditemukan
         Ukuran: min=1344, max=1344, mean=1344
  [3/4] 480 pasangan grup komutatif
         Pasangan lolos filter: 758,580,480 / 13,005,538,560
         Reduksi: 94.2% tereliminasi
  [4/4] Streaming batch max-plus check (batch_size=2,000)...
         8,127,840/758,580,480 (1.1%) | 14s elapsed | ETA ~1316s | 1088 found
         19,868,352/758,580,480 (2.6%) | 26s elapsed | ETA ~971s | 2048 found
         26,189,856/758,580,480 (3.5%) | 37s elapsed | ETA ~1022s | 2924 found
         32,511,360/758,580,480 (4.3%) | 47s elapsed | ETA ~1047s | 7448 found
         38,832,864/758,580,480 (5.1%) | 57s elapsed | ETA ~1059s | 19472 found
         46,058,208/758,580,480 (6.1%) | 68s elapsed | ETA ~1058s | 30896 found
         58,701,216/758,580,480 (7.7%) | 81s elapsed | ETA ~961s | 33500 found
     

In [50]:
# ============================================================
# Export Pasangan Komutatif ke LaTeX (format multicol)
# ============================================================

def export_commutative_pairs(pairs: List[dict], 
                              filename: str,
                              order: int,
                              output_dir: str = ".",
                              num_cols: int = 2) -> str:
    """Export pasangan komutatif ke file .tex dengan format multicol"""
    lines = []
    
    # Matrix to latex helper (compact)
    def mat_to_tex(m):
        rows = " \\\\ ".join(" & ".join(str(int(m[i,j])) for j in range(m.shape[1])) for i in range(m.shape[0]))
        return f"\\begin{{pmatrix}} {rows} \\end{{pmatrix}}"
    
    # Permutation to matrix format (vertical stacking, scriptsize)
    def perms_to_matrix_small(perms: dict, symbols: list) -> str:
        rows = " \\\\ ".join(
            f"\\tau_{{{s}}}={PermutationExtractor.permutation_to_cycle_notation(perms[s])}"
            for s in symbols
        )
        return f"\\begin{{matrix}} {rows} \\end{{matrix}}"
    
    # Header
    lines.append(f"% File: {filename}.tex")
    lines.append(f"% Commutative pairs of Latin squares order {order}")
    lines.append(f"% Total: {len(pairs)} pairs")
    lines.append("")
    
    lines.append(f"\\subsection*{{Pasangan Latin Square Komutatif Order {order}}}")
    lines.append("")
    lines.append(f"Terdapat {len(pairs)} pasangan $(A, B)$ dengan $A \\neq B$ yang memenuhi $A \\otimes B = B \\otimes A$:")
    lines.append("")
    lines.append(f"\\begin{{multicols}}{{{num_cols}}}")
    lines.append("\\small")
    lines.append("")
    
    for idx, pair in enumerate(pairs, 1):
        A = pair['A']
        B = pair['B']
        product = pair['product']
        
        # Ekstrak permutasi
        perms_A = PermutationExtractor.extract_permutations(A)
        perms_B = PermutationExtractor.extract_permutations(B)
        
        # Format permutasi (scriptsize matrix)
        perm_A_matrix = perms_to_matrix_small(perms_A, A.symbols)
        perm_B_matrix = perms_to_matrix_small(perms_B, B.symbols)
        
        lines.append(f"\\noindent\\textbf{{{idx}.}} $L_{{{pair['A_index']}}} \\circ L_{{{pair['B_index']}}}$")
        lines.append("")
        # Matriks dan permutasi di samping (satu baris)
        lines.append(f"$A = {mat_to_tex(A.matrix)} \\quad {perm_A_matrix}$")
        lines.append("")
        lines.append(f"$B = {mat_to_tex(B.matrix)} \\quad {perm_B_matrix}$")
        lines.append("")
        lines.append(f"$A \\otimes B = {mat_to_tex(product)}$")
        lines.append("")
        lines.append("\\vspace{0.3em}")
        lines.append("\\hrule")
        lines.append("\\vspace{0.3em}")
        lines.append("")
    
    lines.append("\\end{multicols}")
    
    # Write to file
    filepath = os.path.join(output_dir, f"{filename}.tex")
    with open(filepath, 'w', encoding='utf-8') as f:
        f.write("\n".join(lines))
    
    return filepath

# Export (2 kolom untuk order 3, bisa ubah num_cols sesuai kebutuhan)
output_file_comm = f"latin_square_commutative_pairs_order_{ORDER}"
filepath_comm = export_commutative_pairs(
    commutative_pairs,
    filename=output_file_comm,
    order=ORDER,
    num_cols=3
)

print(f"✓ File LaTeX berhasil dibuat: {filepath_comm}")
print(f"  Total: {len(commutative_pairs)} pasangan komutatif")
print(f"\nUntuk import di main.tex, gunakan:")
print(f"  \\input{{{output_file_comm}}}")

✓ File LaTeX berhasil dibuat: .\latin_square_commutative_pairs_order_5.tex
  Total: 460140 pasangan komutatif

Untuk import di main.tex, gunakan:
  \input{latin_square_commutative_pairs_order_5}
